# 03 — Vektorisierung: Von Text zu Zahlen

**Ziel dieses Notebooks:** Den vorverarbeiteten Text (`train_preprocessed.csv`)
in numerische Feature-Vektoren ueberfuehren, die ein klassisches ML-Modell
(Phase 4) verarbeiten kann. Methoden, Parameter und Textvariante werden
datengetrieben verglichen, nicht angenommen.

**Leitfragen:**
- BoW vs. TF-IDF: welcher Unterschied zeigt sich konkret?
- Wie beeinflussen min_df/max_df die Vokabulargroesse?
- Helfen n-Gramme (Bigramme/Trigramme) ueber Unigramme hinaus?
- Welche der vier Preprocessing-Varianten (clean/no_stopwords/stemmed/
  lemmatized) eignet sich am besten als Vektorisierungs-Basis?
- Wie aehnlich/unaehnlich sind Tweets im Vektorraum (Cosine Similarity)?
- Reduziert LSA/SVD sinnvoll, ohne viel Varianz zu verlieren?
- Wie gut trennen sich die Klassen visuell im reduzierten Raum
  (PCA/t-SNE, 2D) und quantifiziert (Silhouette-Score)?

**Wichtige Einschraenkung:** Die finale Entscheidung "welche Variante/
Parameter performt am besten" wird hier NICHT abschliessend getroffen —
nur Indikatoren erhoben (Dimensionen, Sparsity, Separierbarkeit). Der
belastbare Test mit echtem Modelltraining folgt in Phase 4.

**Output:** finaler Vectorizer + Matrix (fuer Phase 4), Plots/Tabellen
wie gewohnt.

In [ ]:
# =============================================================================
# Zelle 02 – Imports, Store44-Style aktivieren, vorverarbeiteten Datensatz laden
# =============================================================================
# keep_default_na=False: keine automatische NaN-Erkennung beim Einlesen.
# Leere Strings in Text-Spalten sind gueltige Werte (Tweet bestand nur aus
# Stopwords/Mentions). keyword/location werden danach gezielt zurueck zu NaN
# konvertiert, da dort Fehlen inhaltlich bedeutungsvoll ist (Notebook 01).

import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from viz_config import (
    apply_store44_style, save_figure,
    COLOR_GOLD, COLOR_BLUE, COLOR_GREEN,
    COLOR_CLASS_0, COLOR_CLASS_1, COLOR_TEXT_MUTED
)

apply_store44_style()

train = pd.read_csv(
    "../data/processed/train_preprocessed.csv",
    keep_default_na=False,
)

train[["keyword", "location"]] = train[["keyword", "location"]].replace("", np.nan)

text_columns = ["text", "text_clean", "text_no_stopwords", "text_stemmed", "text_lemmatized"]
print("Shape:", train.shape)
print("\nNaN-Check Text-Spalten (sollte 0 sein):")
print(train[text_columns].isnull().sum())
print("\nNaN-Check keyword/location (echte Fehlquote, sollte wieder Notebook-01-Werten entsprechen):")
print(train[["keyword", "location"]].isnull().sum())

train[text_columns].head(3)

In [ ]:
# =============================================================================
# Zelle 03 – Bag of Words (CountVectorizer) als Ausgangspunkt
# =============================================================================
# Wir starten mit text_no_stopwords als Arbeitsbasis (neutraler Mittelweg
# zwischen roh und stark transformiert). Vergleich aller 4 Varianten folgt
# spaeter in Zelle 11.

from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(train["text_no_stopwords"])

print("Shape der BoW-Matrix:", X_bow.shape)
print("Vokabulargroesse:", len(bow_vectorizer.vocabulary_))

# Sparsity: wie viel Prozent der Matrix sind Nullen?
n_total = X_bow.shape[0] * X_bow.shape[1]
n_nonzero = X_bow.nnz
sparsity = (1 - n_nonzero / n_total) * 100
print(f"\nNicht-Null-Eintraege: {n_nonzero:,} von {n_total:,} ({100-sparsity:.3f}%)")
print(f"Sparsity (Anteil Nullen): {sparsity:.3f}%")

# Speicherbedarf: dicht vs. sparse (zeigt, warum sparse-Format essenziell ist)
dense_memory_mb = n_total * 8 / (1024**2)  # 8 Bytes pro float64
sparse_memory_mb = X_bow.data.nbytes / (1024**2)
print(f"\nSpeicherbedarf als DICHTE Matrix: {dense_memory_mb:.1f} MB")
print(f"Speicherbedarf als SPARSE Matrix: {sparse_memory_mb:.1f} MB")
print(f"Faktor: {dense_memory_mb/sparse_memory_mb:.0f}x weniger Speicher durch Sparse-Format")

## Erkenntnis: Bag of Words (Ausgangspunkt)

- Matrix-Shape: 7.434 x 15.803 (Dokumente x Vokabular)
- Sparsity: 99,944% — nur 0,056% der Matrix sind Nicht-Null-Eintraege
- Speicherersparnis durch Sparse-Format: Faktor 1.787x (896,3 MB dicht
  vs. 0,5 MB sparse fuer identischen Inhalt)

**Einordnung:** Die extreme Sparsity ist strukturell erwartbar — jeder
einzelne Tweet enthaelt nur ~9 Woerter (siehe Notebook 02, nach Stopword-
Entfernung), das Gesamtvokabular aber 15.803 Woerter. Jede Zeile der
Matrix hat daher fast ausschliesslich Nullen. Das ist der zentrale Grund,
warum Textverarbeitung mit sparse-Datenstrukturen arbeitet statt mit
regulaeren Arrays — Letzteres waere bei diesem Datensatz bereits jetzt
unhandlich, bei groesseren Korpora schlicht nicht mehr durchfuehrbar.

**Limitation von BoW:** Zaehlt nur Worthaeufigkeiten, gewichtet jedes
Wort gleich — ein 50x vorkommendes, thematisch neutrales Wort (z. B.
"like") hat denselben Einfluss wie ein seltenes, hoch spezifisches Wort
(z. B. "derailment"). TF-IDF (naechste Zelle) adressiert genau das.

In [ ]:
# =============================================================================
# Zelle 05 – TF-IDF (Term Frequency-Inverse Document Frequency)
# =============================================================================

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(train["text_no_stopwords"])

print("Shape der TF-IDF-Matrix:", X_tfidf.shape)
print("Vokabulargroesse:", len(tfidf_vectorizer.vocabulary_))

# Gleiche Dimensionen wie BoW erwartet (gleicher Tokenizer/Vokabular-Aufbau,
# nur die GEWICHTUNG der Eintraege unterscheidet sich)
print("\nShape identisch zu BoW?", X_tfidf.shape == X_bow.shape)

# Konkreter Vergleich an einem Beispieldokument: Rohzaehlung vs. TF-IDF-Gewicht
doc_idx = 2
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

bow_row = X_bow[doc_idx].toarray().flatten()
tfidf_row = X_tfidf[doc_idx].toarray().flatten()

nonzero_idx = bow_row.nonzero()[0]
comparison_df = pd.DataFrame({
    "word": feature_names[nonzero_idx],
    "bow_count": bow_row[nonzero_idx],
    "tfidf_weight": tfidf_row[nonzero_idx].round(3),
}).sort_values("tfidf_weight", ascending=False)

print(f"\nText: {train['text_no_stopwords'].iloc[doc_idx]}")
print("\nBoW-Zaehlung vs. TF-IDF-Gewicht (sortiert nach TF-IDF):")
print(comparison_df.to_string(index=False))

## Erkenntnis: TF-IDF vs. BoW

Gleiche Matrix-Dimensionen (7.434 x 15.803) — der Unterschied liegt
ausschliesslich in der Gewichtung, nicht im Vokabular.

**Beispiel-Tweet** ("residents asked shelter place notified officers
evacuation shelter place orders expected"):
- "shelter" (2x) erhaelt das hoechste TF-IDF-Gewicht (0,555) — sowohl
  haeufig im Tweet als auch vermutlich seltener im Gesamtkorpus
- "evacuation" hat trotz inhaltlicher Relevanz nur 0,211 — kommt im
  Gesamtkorpus offenbar haeufiger vor (in vielen Disaster-Tweets praesent),
  daher niedrigeres IDF-Gewicht trotz thematischer Bedeutung
- Selbst bei gleichem BoW-Count (1) unterscheiden sich die TF-IDF-Gewichte
  deutlich (0,315 vs. 0,211) — reiner IDF-Effekt (Seltenheit im Korpus)

**Konsequenz:** TF-IDF gewichtet automatisch herunter, was im Korpus
verbreitet ist (auch wenn thematisch relevant), und hebt hervor, was
selten/spezifisch ist. Das ist meist vorteilhaft fuer Klassifikation,
kann aber theoretisch auch thematisch wichtige, aber haeufige Begriffe
unterbewerten — wird in Phase 4 empirisch geprueft (TF-IDF vs. BoW als
Modell-Input im direkten Vergleich).

**Entscheidung:** TF-IDF wird als primaere Vektorisierungsmethode
fuer Phase 4 verwendet (Standardwahl, theoretisch begruendet); BoW
bleibt als Vergleichsoption fuer die Ablationsstudie verfuegbar.

In [ ]:
# =============================================================================
# Zelle 07 – min_df/max_df: Effekt auf Vokabulargroesse
# =============================================================================
# min_df: Wort muss in mindestens N Dokumenten vorkommen (filtert Raritaeten)
# max_df: Wort darf in hoechstens X% der Dokumente vorkommen (filtert
# zu haeufige, wenig spezifische Woerter)

min_df_values = [1, 2, 3, 5, 10, 20]
max_df_values = [1.0, 0.9, 0.7, 0.5, 0.3]

min_df_results = []
for val in min_df_values:
    vec = TfidfVectorizer(min_df=val)
    X = vec.fit_transform(train["text_no_stopwords"])
    min_df_results.append({"min_df": val, "vocab_size": X.shape[1]})

max_df_results = []
for val in max_df_values:
    vec = TfidfVectorizer(max_df=val)
    X = vec.fit_transform(train["text_no_stopwords"])
    max_df_results.append({"max_df": val, "vocab_size": X.shape[1]})

min_df_df = pd.DataFrame(min_df_results)
max_df_df = pd.DataFrame(max_df_results)

print("=== min_df-Effekt ===")
print(min_df_df)
print("\n=== max_df-Effekt ===")
print(max_df_df)

min_df_df.to_csv("../reports/tables/03_min_df_effect.csv", index=False)
max_df_df.to_csv("../reports/tables/03_max_df_effect.csv", index=False)

# Plot: beide Effekte nebeneinander
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(min_df_df["min_df"], min_df_df["vocab_size"], marker="o", color=COLOR_GOLD)
axes[0].set_xlabel("min_df (Mindestanzahl Dokumente)")
axes[0].set_ylabel("Vokabulargroesse")
axes[0].set_title("Effekt von min_df")

axes[1].plot(max_df_df["max_df"], max_df_df["vocab_size"], marker="o", color=COLOR_BLUE)
axes[1].set_xlabel("max_df (max. Dokumentanteil)")
axes[1].set_ylabel("Vokabulargroesse")
axes[1].set_title("Effekt von max_df")
axes[1].invert_xaxis()

fig.suptitle("Vokabulargroesse in Abhaengigkeit von min_df/max_df")
fig.tight_layout()

save_figure(fig, "../reports/figures/03_min_max_df_effect.png")
plt.show()

## Erkenntnis: min_df/max_df-Effekt

**min_df: starker, schnell abflachender Effekt**
- min_df=1 (Standard): 15.803 Woerter
- min_df=2: 6.074 Woerter (-61,6%) — bereits die Haelfte der Reduktion
  durch einen einzigen Schritt erreicht
- min_df=20: nur noch 697 Woerter

Bestaetigt den Heaps'-Law-Befund aus Notebook 01: ein Grossteil des
Vokabulars besteht aus seltenen Einzelvorkommen (84,3% der location-Werte
waren Einzelfaelle, hier zeigt sich ein analoges Muster im Text-Vokabular).

**max_df: praktisch wirkungslos bis 0,3**
Vokabulargroesse bleibt bei 15.803 konstant ueber den gesamten getesteten
Bereich. Erklaerung: Tweets sind sehr kurz (durchschnittlich 9,2 Tokens
nach Stopword-Entfernung, siehe Notebook 02) — kein Wort kommt in mehr
als 30-50% aller 7.434 Dokumente vor. max_df waere erst bei deutlich
laengeren Dokumenten oder einem kleineren, thematisch engeren Korpus
relevant.

**Konsequenz fuer Phase 4:** min_df ist der Hyperparameter mit
tatsaechlichem Einfluss und gehoert ins Grid-Search-Tuning (Phase 5).
max_df kann auf Standardwert (1.0, kein Filter) belassen werden — der
Suchraum sollte sich nicht mit einem wirkungslosen Parameter aufblaehen.

In [ ]:
# =============================================================================
# Zelle 09 – n-Gramme: Unigramm vs. Bigramm vs. Trigramm
# =============================================================================
# n-Gramme erfassen Wortfolgen statt einzelner Woerter, z. B. Bigramm
# "forest fire" als ein Feature statt nur "forest" und "fire" getrennt.

ngram_configs = [
    ("Unigramm (1,1)", (1, 1)),
    ("Bigramm (2,2)", (2, 2)),
    ("Trigramm (3,3)", (3, 3)),
    ("Uni+Bigramm (1,2)", (1, 2)),
    ("Uni+Bi+Trigramm (1,3)", (1, 3)),
]

ngram_results = []
for name, ngram_range in ngram_configs:
    vec = TfidfVectorizer(ngram_range=ngram_range)
    X = vec.fit_transform(train["text_no_stopwords"])
    ngram_results.append({"config": name, "vocab_size": X.shape[1]})

ngram_df = pd.DataFrame(ngram_results)
print(ngram_df)
ngram_df.to_csv("../reports/tables/03_ngram_effect.csv", index=False)

# Beispiel: welche Bigramme sind am haeufigsten?
vec_bigram = TfidfVectorizer(ngram_range=(2, 2), min_df=5)
X_bigram = vec_bigram.fit_transform(train["text_no_stopwords"])
bigram_counts = np.asarray(X_bigram.sum(axis=0)).flatten()
bigram_names = np.array(vec_bigram.get_feature_names_out())
top_bigrams = pd.DataFrame({"bigram": bigram_names, "tfidf_sum": bigram_counts}) \
    .sort_values("tfidf_sum", ascending=False).head(15)
print("\nTop 15 Bigramme (nach TF-IDF-Summe, min_df=5):")
print(top_bigrams.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(ngram_df["config"], ngram_df["vocab_size"], color=COLOR_GOLD)
ax.set_ylabel("Vokabulargroesse")
ax.set_title("Vokabulargroesse je n-Gramm-Konfiguration")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()

save_figure(fig, "../reports/figures/03_ngram_effect.png")
plt.show()

## Erkenntnis: n-Gramme

**Vokabular waechst stark mit hoeherer n-Gramm-Ordnung:**
| Konfiguration | Vokabulargroesse | Faktor ggue. Unigramm |
|---|---|---|
| Unigramm | 15.803 | 1,0x |
| Bigramm | 45.660 | 2,9x |
| Trigramm | 43.824 | 2,8x |
| Uni+Bigramm | 61.463 | 3,9x |
| Uni+Bi+Trigramm | 105.287 | 6,7x |

Klassische Kombinationsexplosion: Bigramme/Trigramme erzeugen deutlich
mehr eindeutige Kombinationen als Unigramme, bei nur 7.434 kurzen Tweets
(~9 Tokens/Tweet) bedeutet das viele Bigramme mit sehr wenigen
Vorkommen (hohe Sparsity, vermutlich aehnlich dem min_df-Befund aus
Zelle 08).

**Top-Bigramme zeigen gemischtes Bild:**
- Inhaltlich aussagekraeftig: "burning buildings", "natural disaster",
  "mass murderer", "suicide bombing", "wild fires", "dust storm",
  "first responders" — echte, thematisch praezise Wortpaare
- Rauschen/Artefakt: **"num num"** an erster Stelle (TF-IDF-Summe 177,
  mit Abstand hoechster Wert) — entsteht durch den NUM-Platzhalter aus
  Notebook 02 (z. B. bei Datums-/Zahlenfolgen wie "8/6/2015" oder
  Telefonnummern, die mehrere aufeinanderfolgende Zahlenblöcke enthalten).
  Traegt keine inhaltliche Bedeutung — analog zum frueheren URL-Befund.
- Themenfremd: "youtube video", "liked youtube" — automatisierte
  Social-Media-Tweets ohne Disaster-Bezug

**Konsequenz:** n-Gramme liefern potenziell wertvolle zusaetzliche
Information (mehrwortige Fachbegriffe), aber auf Kosten stark erhoehter
Dimensionalitaet und mit mindestens einem klaren Artefakt ("num num").
Ob sich der Mehrwert in echter Modellperformance niederschlaegt, wird in
Phase 4 getestet (Unigramm vs. Uni+Bigramm als Ablationsvergleich), nicht
hier vorab entschieden.

**Offener Punkt:** "num num"-Artefakt koennte durch Ausschluss von "num"
aus n-Gramm-Bildung behoben werden (z. B. eigene Stopword-Erweiterung)
— wird vermerkt, aber nicht in Phase 3 behoben, da es sich nur auf
Bigramm/Trigramm-Konfigurationen auswirkt, die ohnehin erst in Phase 4
final getestet werden.

In [ ]:
# =============================================================================
# Zelle 11 – Vergleich der vier Preprocessing-Varianten als Vektorisierungs-Basis
# =============================================================================
# Vorlaeufiger Indikator (Dimensionen), KEIN Performance-Beweis -
# echter Test folgt in Phase 4 als Ablationsstudie.

text_variants = {
    "text_clean": train["text_clean"],
    "text_no_stopwords": train["text_no_stopwords"],
    "text_stemmed": train["text_stemmed"],
    "text_lemmatized": train["text_lemmatized"],
}

variant_results = []
for name, series in text_variants.items():
    vec = TfidfVectorizer()
    X = vec.fit_transform(series)
    variant_results.append({
        "variant": name,
        "vocab_size": X.shape[1],
        "avg_nonzero_per_doc": round(X.nnz / X.shape[0], 2),
    })

variant_df = pd.DataFrame(variant_results)
print(variant_df)
variant_df.to_csv("../reports/tables/03_text_variant_comparison.csv", index=False)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(variant_df["variant"], variant_df["vocab_size"], color=COLOR_GOLD)
ax.set_ylabel("Vokabulargroesse (TF-IDF)")
ax.set_title("Vokabulargroesse je Text-Variante")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()

save_figure(fig, "../reports/figures/03_text_variant_comparison.png")
plt.show()

## Erkenntnis: Vergleich der Text-Varianten als TF-IDF-Input

| Variante | Vokabulargroesse | Nonzero/Tweet |
|---|---|---|
| text_clean | 16.516 | 12,82 |
| text_no_stopwords | 15.803 | 8,84 |
| text_stemmed | 12.832 | 8,81 |
| text_lemmatized | 14.568 | 8,80 |

**Bestaetigt die Erwartung aus Notebook 02:** Reihenfolge der
Vokabulargroesse entspricht exakt der dort gemessenen (text_clean >
text_no_stopwords > text_lemmatized > text_stemmed).

**avg_nonzero_per_doc bestaetigt den Stopword-Effekt klar:** text_clean
liegt deutlich hoeher (12,82) als die anderen drei (~8,8) — ausschliesslich
durch die noch enthaltenen Stopwords, die als zusaetzliche Nicht-Null-
Eintraege pro Tweet zaehlen, aber kaum Unterscheidungskraft liefern
(bereits in Notebook 01, Zelle 09c belegt: nur 3,4% der Klassen-
Ueberschneidung waren Stopwords).

**Einordnung:** text_clean scheidet als Kandidat fuer Phase 4 aus
diesem Grund vermutlich aus (unnoetig hohe Dimensionalitaet ohne
Mehrwert). Die Entscheidung zwischen text_no_stopwords, text_stemmed
und text_lemmatized bleibt offen — alle drei sind nach diesem
vorlaeufigen Indikator naeherungsweise gleichwertig in der Informations-
dichte (Nonzero/Tweet fast identisch), unterscheiden sich aber in
Vokabulargroesse und Interpretierbarkeit (siehe Notebook 02, Zelle 11).

**Finale Entscheidung folgt in Phase 4** als Ablationsstudie (alle drei
Varianten mit identischem Modell trainiert, F1 per Cross-Validation
verglichen) — konsistent mit der durchgaengigen Einschraenkung dieses
Notebooks.

In [ ]:
# =============================================================================
# Zelle 13 – Cosine Similarity zwischen TF-IDF-Vektoren
# =============================================================================
# Misst den Winkel zwischen zwei Vektoren (0 = orthogonal/unaehnlich,
# 1 = identische Richtung/sehr aehnlich), unabhaengig von der Textlaenge.

from sklearn.metrics.pairwise import cosine_similarity

# Fall 1: bekannte Near-Duplicates aus Notebook 01 (Zelle 13/14) -
# "Ted Cruz fires back..." kam mehrfach mit unterschiedlicher Kurz-URL vor
ted_cruz_mask = train["text"].str.contains("Ted Cruz fires back", na=False)
ted_cruz_idx = train[ted_cruz_mask].index.tolist()
print(f"Gefundene 'Ted Cruz fires back'-Tweets: {len(ted_cruz_idx)}")

if len(ted_cruz_idx) >= 2:
    sim_matrix = cosine_similarity(X_tfidf[ted_cruz_idx])
    print("\nCosine Similarity zwischen diesen Near-Duplicates:")
    print(pd.DataFrame(sim_matrix, index=ted_cruz_idx, columns=ted_cruz_idx).round(3))

# Fall 2: ein beliebiger Disaster-Tweet vs. die 5 aehnlichsten Tweets im Korpus
anchor_idx = train[train["target"] == 1].index[10]
anchor_text = train.loc[anchor_idx, "text_no_stopwords"]
print(f"\n=== Anchor-Tweet (Index {anchor_idx}) ===")
print(anchor_text)

sims_to_anchor = cosine_similarity(X_tfidf[anchor_idx], X_tfidf).flatten()
top5_idx = sims_to_anchor.argsort()[::-1][1:6]  # [0] ist der Tweet selbst, daher ab [1]

print("\nTop 5 aehnlichste Tweets:")
for idx in top5_idx:
    print(f"  Sim={sims_to_anchor[idx]:.3f} | {train.loc[idx, 'text_no_stopwords']}")

# Fall 3: zwei zufaellige, thematisch unterschiedliche Tweets als Kontrast
random_idx = train.sample(2, random_state=7).index.tolist()
sim_random = cosine_similarity(X_tfidf[random_idx[0]], X_tfidf[random_idx[1]])[0][0]
print(f"\n=== Kontrast: zwei zufaellige Tweets ===")
print(f"Tweet A: {train.loc[random_idx[0], 'text_no_stopwords']}")
print(f"Tweet B: {train.loc[random_idx[1], 'text_no_stopwords']}")
print(f"Cosine Similarity: {sim_random:.3f}")

In [ ]:
# =============================================================================
# Zelle 13b – Originaltexte der 'Ted Cruz'-Near-Duplicates (Kontext zur
# Similarity-Matrix aus Zelle 13)
# =============================================================================

print("Originaltexte (vor Preprocessing):\n")
for idx in ted_cruz_idx:
    print(f"[{idx}] {train.loc[idx, 'text']}")

print("\nNach Preprocessing (text_no_stopwords, Basis fuer Cosine Similarity):\n")
for idx in ted_cruz_idx:
    print(f"[{idx}] {train.loc[idx, 'text_no_stopwords']}")

## Erkenntnis: Cosine Similarity

**Near-Duplicate-Bestaetigung:** Die vier "Ted Cruz fires back"-Tweets
(Notebook 01, Zelle 13/14 erstmals beobachtet) zeigen nach Preprocessing
exakte Similarity 1,0 — Originaltexte unterschieden sich nur in der URL,
die in Notebook 02 vollstaendig entfernt wurde. Im Vektorraum sind diese
vier Tweets nun ununterscheidbar.

**Konsequenz:** Was in Notebook 01 als "technisches Duplikat-Risiko" nur
vermerkt wurde (keine Aktion), zeigt sich hier konkret: Diese vier Zeilen
verhalten sich im Modell wie ein 4-faches Gewicht fuer denselben Tweet.
Bei einem zufaelligen Train/Validation-Split koennten 2-3 Kopien im
Training und 1 in der Validierung landen — das waere Data Leakage
(identischer Vektor in Training UND Validierung). Sollte vor Phase 4
behoben werden (siehe offener Punkt unten).

**Thematische Aehnlichkeit (Anchor-Tweet "three people died heat wave"):**
Top-Treffer sind ueberwiegend echte Heat-Wave-Tweets (Sim 0,28-0,39),
aber bereits bei Rang 2-3 mischen sich thematisch unpassende Tweets
("heat wave squad revitup pizzarev" - vermutlich Musik/Slang-Kontext).
Aehnlichkeitswerte unter 0,4 sind insgesamt moderat, kein Tweet ist
einem anderen extrem aehnlich (ausser den echten Duplikaten oben) -
zeigt die hohe lexikalische Vielfalt kurzer Tweets.

**Kontrast-Tweets:** Similarity = 0,000 zwischen zwei thematisch
unverwandten Tweets (kein gemeinsames Vokabular) — erwartbares Verhalten,
bestaetigt dass TF-IDF-Vektoren bei Tweets ohne Wortueberschneidung
tatsaechlich orthogonal sind.

## Offener Punkt — Near-Duplicate-Bereinigung

In Notebook 01 nur vermerkt, hier konkret bestaetigt: Tweets, die sich
NUR in der URL unterscheiden, werden nach Preprocessing identisch.
Empfehlung: vor Phase 4 zusaetzliche Bereinigung auf Basis von
text_no_stopwords (oder text_clean) statt nur des Original-Texts -
Duplikate, die ERST nach Preprocessing entstehen, wurden in Notebook 01
noch nicht erfasst (dort wurde nur auf rohem `text` geprueft).

In [ ]:
# =============================================================================
# Zelle 13c – Systematische Pruefung: Duplikate/Konflikte entstehen erst
# nach text_clean-Normalisierung (URL-Entfernung, Lowercase)
# =============================================================================

cols = ["text", "text_clean", "text_no_stopwords", "text_stemmed", "text_lemmatized", "target"]

# Duplikatgruppen auf text_clean-Basis identifizieren
dupes_clean = train[train.duplicated(subset=["text_clean"], keep=False)].copy()
conflict_groups = dupes_clean.groupby("text_clean")["target"].nunique()
conflict_texts = conflict_groups[conflict_groups > 1].index.tolist()
harmless_texts = conflict_groups[conflict_groups == 1].index.tolist()

# Umfang
print(f"Betroffene Zeilen gesamt: {len(dupes_clean)}")
print(f"Gruppen gesamt: {dupes_clean['text_clean'].nunique()}")
print(f"Davon Konflikt-Gruppen (widersprüchliche Labels): {len(conflict_texts)}")
print(f"Davon harmlose Duplikat-Gruppen (gleiche Labels): {len(harmless_texts)}")

print("\n" + "=" * 70)
print("BEISPIEL A — KONFLIKT: gleicher text_clean, unterschiedliche Labels")
print("=" * 70)
for _, row in dupes_clean[dupes_clean["text_clean"] == conflict_texts[0]][cols].iterrows():
    print(f"\n  target={row['target']}")
    for col in cols[:-1]:
        print(f"  {col:20s}: {row[col]}")

print("\n" + "=" * 70)
print("BEISPIEL B — HARMLOS: gleicher text_clean, identische Labels")
print("=" * 70)
for _, row in dupes_clean[dupes_clean["text_clean"] == harmless_texts[0]][cols].iterrows():
    print(f"\n  target={row['target']}")
    for col in cols[:-1]:
        print(f"  {col:20s}: {row[col]}")

## Erkenntnis: Cosine Similarity & Duplikate nach Preprocessing

**Cosine Similarity — Verhalten im Vektorraum:**
- Near-Duplicates (gleicher Text, unterschiedliche URL): Similarity = exakt
  1,0 nach Preprocessing — URL war der einzige Unterschied, nach
  text_clean-Normalisierung ununterscheidbar
- Thematisch aehnliche Tweets: Similarity 0,28-0,39 (moderat) — zeigt
  die hohe lexikalische Vielfalt kurzer Tweets, kein Tweet ist einem
  anderen extrem aehnlich (ausser echten Duplikaten)
- Thematisch unverwandte Tweets: Similarity = 0,000 (orthogonal im
  Vektorraum, kein gemeinsames Vokabular)

**Systematische Duplikat-Pruefung nach text_clean-Normalisierung:**
Notebook 01 pruefte Duplikate/Konflikte nur auf Roh-Text-Ebene (18
Konflikt-Gruppen gefunden und entfernt). Nach text_clean-Normalisierung
(URL-Entfernung, Lowercase, Mojibake, NUM-Platzhalter) entstehen weitere
Duplikate:

| | Gruppen | Betroffene Zeilen |
|---|---|---|
| Konflikt-Gruppen (widersprueche Labels) | 52 | ~364 |
| Harmlose Duplikat-Gruppen (gleiche Labels) | 193 | ~462 |
| Gesamt | 245 | 826 |

**Ursachen-Analyse der 52 Konflikt-Gruppen:**
- 52 von 52 (100%) entstehen ausschliesslich durch URL-Normalisierung:
  derselbe Tweet-Text wurde mit verschiedenen URLs geteilt, manchmal
  von verschiedenen Accounts mit unterschiedlichem Label versehen
- Keine Konflikt-Gruppe ohne URL-Beteiligung — d. h. kein genuines
  inhaltliches Label-Rauschen auf text_clean-Ebene (das wurde bereits
  in Notebook 01 vollstaendig entfernt)

**Nebeneffekt des NUM-Platzhalters:**
Inhaltlich verschiedene Ereignisse (z. B. zwei aufeinanderfolgende
Erdbeben-Reports M4.0 und M3.8 desselben Messsystems) werden durch
NUM-Platzhalter-Ersetzung nach Preprocessing identisch — harmlos, da
Labels identisch sind (kein Data Leakage), aber zeigt die konzeptionelle
Grenze des Platzhalter-Ansatzes: exakte Zahlen verlieren ihre
unterscheidende Bedeutung.

**Konsequenz — zwei unterschiedliche Loesungen fuer zwei Ursachen:**
1. URL-bedingte Konflikt-Gruppen: Duplikat-/Konflikt-Bereinigung muss
   auf text_clean-Basis erfolgen, nicht nur auf Roh-Text. Korrektur
   erfolgt in Notebook 01 (neue Analyse-Zelle) und Notebook 02 (neue
   Bereinigungszelle nach text_clean-Erzeugung). train_clean.csv
   und train_preprocessed.csv werden danach neu generiert.
2. NUM-bedingte harmlose Duplikate: keine Bereinigung noetig — gleiche
   Labels, kein Data-Leakage-Risiko.